# SAM2 AUTO BOX VS AGRICLIP
"" POINT PROMPTS

In [37]:
# ============================================================
# CELL 1 — IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms

from datasets import load_dataset

print("=" * 70)
print("AGRICLIP + SAM2 PROMPT STRATEGY NOTEBOOK")
print("=" * 70)

print("PyTorch:", torch.__version__)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)

AGRICLIP + SAM2 PROMPT STRATEGY NOTEBOOK
PyTorch: 2.12.0+cu130
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 4090


In [38]:
# ============================================================
# CELL 2 — PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/zahranadeem/thesis"
)

AGRICLIP_ROOT = (
    PROJECT_ROOT
    / "agriclip_sam2"
    / "AgriCLIP"
)

AGRICLIP_ALIGNMENT_DIR = (
    AGRICLIP_ROOT
    / "AgriCLIP_alignment"
)

# ------------------------------------------------------------
# DINO
# ------------------------------------------------------------

DINO_CANDIDATES = [
    PROJECT_ROOT
    / "agriclip_Zahra"
    / "weights"
    / "DINO-PT.pth",

    PROJECT_ROOT
    / "agriclip_sam2"
    / "AgriCLIP"
    / "Weights"
    / "DINO-PT.pth",

    PROJECT_ROOT
    / "DINO-PT.pth"
]

DINO_PATH = None

for candidate in DINO_CANDIDATES:
    if candidate.exists():
        DINO_PATH = candidate
        break

# ------------------------------------------------------------
# AgriCLIP aligner
# ------------------------------------------------------------

ALIGNER_PATH = (
    PROJECT_ROOT
    / "vegann_aligner.pth"
)

# ------------------------------------------------------------
# Original non-augmented fine-tuned SAM2
# ------------------------------------------------------------

SAM2_CHECKPOINT = (
    PROJECT_ROOT
    / "sam2_vegann_checkpoints"
    / "best_sam2_vegann_mask_decoder.pt"
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_NAME = (
    "facebook/sam2.1-hiera-tiny"
)

print("=" * 70)
print("PATH CHECK")
print("=" * 70)

print("Project:")
print(PROJECT_ROOT)

print("\nAgriCLIP:")
print(AGRICLIP_ROOT)

print("\nDINO:")
print(DINO_PATH)

print("\nAligner:")
print(ALIGNER_PATH)

print("\nSAM2:")
print(SAM2_CHECKPOINT)

print("\nSAM2 exists:",
      SAM2_CHECKPOINT.exists())

print("Aligner exists:",
      ALIGNER_PATH.exists())

print("DINO exists:",
      DINO_PATH is not None and DINO_PATH.exists())

if DINO_PATH is None:
    raise FileNotFoundError(
        "DINO-PT.pth could not be found."
    )

assert ALIGNER_PATH.exists()
assert SAM2_CHECKPOINT.exists()

print("\n✓ All required weight files found")

PATH CHECK
Project:
/home/zahranadeem/thesis

AgriCLIP:
/home/zahranadeem/thesis/agriclip_sam2/AgriCLIP

DINO:
/home/zahranadeem/thesis/agriclip_Zahra/weights/DINO-PT.pth

Aligner:
/home/zahranadeem/thesis/vegann_aligner.pth

SAM2:
/home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt

SAM2 exists: True
Aligner exists: True
DINO exists: True

✓ All required weight files found


In [39]:
# ============================================================
# CELL 3 — AGRICLIP PYTHON PATH
# ============================================================

if str(AGRICLIP_ALIGNMENT_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(AGRICLIP_ALIGNMENT_DIR)
    )

print("=" * 70)
print("AGRICLIP MODULE PATH")
print("=" * 70)

print(
    AGRICLIP_ALIGNMENT_DIR
)

print(
    "\n✓ AgriCLIP alignment directory added to sys.path"
)

AGRICLIP MODULE PATH
/home/zahranadeem/thesis/agriclip_sam2/AgriCLIP/AgriCLIP_alignment

✓ AgriCLIP alignment directory added to sys.path


In [40]:
# ============================================================
# CELL 4 — IMPORT TEXTTOCONCEPT
# ============================================================

from TextToConcept import (
    TextToConcept,
    ClipZeroShot
)

print("=" * 70)
print("TEXTTOCONCEPT")
print("=" * 70)

print(
    "TextToConcept:",
    TextToConcept
)

print(
    "ClipZeroShot:",
    ClipZeroShot
)

print("=" * 70)

TEXTTOCONCEPT
TextToConcept: <class 'TextToConcept.TextToConcept'>
ClipZeroShot: <class 'TextToConcept.ClipZeroShot'>


In [41]:
# ============================================================
# CELL 5 — LOAD VEGANN
# ============================================================

print("=" * 70)
print("LOADING VEGANN")
print("=" * 70)

vegann = load_dataset(
    "simonMadec/VegAnn"
)

print(vegann)

print(
    "\nOriginal dataset size:",
    len(vegann["train"])
)

LOADING VEGANN
DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5'],
        num_rows: 3775
    })
})

Original dataset size: 3775


In [42]:
# ============================================================
# CELL 6 — CREATE EXACT VEGANN TRAIN / VALIDATION SPLIT
# ============================================================

vegann_split = (
    vegann["train"]
    .train_test_split(
        test_size=0.20,
        seed=42
    )
)

train_hf = (
    vegann_split["train"]
)

val_hf = (
    vegann_split["test"]
)

print("=" * 70)
print("VEGANN SPLIT")
print("=" * 70)

print(
    "Training:",
    len(train_hf)
)

print(
    "Validation:",
    len(val_hf)
)

assert len(val_hf) == 755

print(
    "\n✓ Validation split = 755 images"
)

VEGANN SPLIT
Training: 3020
Validation: 755

✓ Validation split = 755 images


In [44]:
# ============================================================
# CELL 7 — VEGANN IMAGE / MASK KEYS
# ============================================================

print("=" * 70)
print("VEGANN COLUMNS")
print("=" * 70)

print(
    "Columns:",
    train_hf.column_names
)

# VegAnn normally uses image/mask.
# Detect automatically so the notebook does not depend
# on a stale variable from another notebook.

IMAGE_KEY = None
MASK_KEY = None

for key in [
    "image",
    "images",
    "img"
]:

    if key in train_hf.column_names:

        IMAGE_KEY = key
        break


for key in [
    "mask",
    "masks",
    "label",
    "labels",
    "segmentation",
    "annotation"
]:

    if key in train_hf.column_names:

        MASK_KEY = key
        break


print(
    "\nIMAGE_KEY:",
    IMAGE_KEY
)

print(
    "MASK_KEY:",
    MASK_KEY
)

if IMAGE_KEY is None:
    raise RuntimeError(
        "Could not find VegAnn image column."
    )

if MASK_KEY is None:
    raise RuntimeError(
        "Could not find VegAnn mask column."
    )

print(
    "\n✓ Dataset keys detected"
)

VEGANN COLUMNS
Columns: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5']

IMAGE_KEY: image
MASK_KEY: mask

✓ Dataset keys detected


In [45]:
# ============================================================
# CELL 8 — IMAGE + MASK HELPERS
# ============================================================

def get_image(
    index,
    split=val_hf
):
    """
    Return a VegAnn image as RGB PIL.Image.
    """

    image = split[index][IMAGE_KEY]

    if isinstance(
        image,
        Image.Image
    ):

        return image.convert("RGB")

    image = np.asarray(
        image
    )

    return Image.fromarray(
        image
    ).convert("RGB")


def get_mask(
    index,
    split=val_hf
):
    """
    Return VegAnn vegetation mask as uint8 binary array.
    """

    mask = split[index][MASK_KEY]

    if isinstance(
        mask,
        Image.Image
    ):

        mask = np.asarray(mask)

    elif torch.is_tensor(mask):

        mask = (
            mask.detach()
            .cpu()
            .numpy()
        )

    else:

        mask = np.asarray(mask)

    if mask.ndim == 3:

        mask = mask[..., 0]

    return (
        mask > 0
    ).astype(np.uint8)


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_image = get_image(0)
test_mask = get_mask(0)

print("=" * 70)
print("IMAGE / MASK TEST")
print("=" * 70)

print(
    "Image type:",
    type(test_image)
)

print(
    "Image size:",
    test_image.size
)

print(
    "Mask shape:",
    test_mask.shape
)

print(
    "Foreground:",
    int(test_mask.sum())
)

print("=" * 70)

IMAGE / MASK TEST
Image type: <class 'PIL.Image.Image'>
Image size: (512, 512)
Mask shape: (512, 512)
Foreground: 208628


In [46]:
# ============================================================
# CELL 9 — LOAD DINO RESNET-50
# ============================================================

print("=" * 70)
print("LOADING DINO RESNET-50")
print("=" * 70)

dino_model = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_resnet50",
    pretrained=True
)

print(
    "DINO model:",
    type(dino_model)
)

print(
    "\nLoading checkpoint:",
    DINO_PATH
)

dino_state_dict = torch.load(
    DINO_PATH,
    map_location="cpu",
    weights_only=True
)

load_result = dino_model.load_state_dict(
    dino_state_dict,
    strict=True
)

print(
    "\nDINO load result:",
    load_result
)

dino_model = (
    dino_model
    .to(device)
)

dino_model.eval()

for parameter in dino_model.parameters():
    parameter.requires_grad = False

print(
    "\n✓ DINO loaded and frozen"
)

print(
    "Layer4:",
    dino_model.layer4
)

print("=" * 70)

LOADING DINO RESNET-50
DINO model: <class 'torchvision.models.resnet.ResNet'>

Loading checkpoint: /home/zahranadeem/thesis/agriclip_Zahra/weights/DINO-PT.pth


Using cache found in /home/zahranadeem/.cache/torch/hub/facebookresearch_dino_main
/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



DINO load result: <All keys matched successfully>

✓ DINO loaded and frozen
Layer4: Sequential(
  (0): Bottleneck(
    (conv1): Conv2d(1024, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv3): Conv2d(512, 2048, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn3): BatchNorm2d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(1024, 2048, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (1): Bottleneck(
    (conv1): Conv2d(2048, 512, kernel_size=(1, 1), str

In [49]:
# ============================================================
# CELL 10 — DINO PREPROCESSING
# ============================================================

dino_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print("=" * 70)
print("DINO TRANSFORM")
print("=" * 70)

print(dino_transform)

print("=" * 70)

DINO TRANSFORM
Compose(
    Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [50]:
# ============================================================
# CELL 11 — DINO LAYER4 HOOK
# ============================================================

captured_feature_map = None


def capture_layer4_output(
    module,
    inputs,
    output
):

    global captured_feature_map

    captured_feature_map = (
        output.detach()
    )


hook_handle = (
    dino_model.layer4.register_forward_hook(
        capture_layer4_output
    )
)

print("=" * 70)
print("DINO LAYER4 HOOK")
print("=" * 70)

print(
    "✓ Hook registered successfully"
)

print("=" * 70)

DINO LAYER4 HOOK
✓ Hook registered successfully


In [51]:
# ============================================================
# CELL 12 — DINO SPATIAL PATCH EXTRACTION
# ============================================================

@torch.no_grad()
def extract_dino_patch_features(
    pil_image
):

    global captured_feature_map

    captured_feature_map = None

    image_tensor = (
        dino_transform(
            pil_image
        )
        .unsqueeze(0)
        .to(device)
    )

    _ = dino_model(
        image_tensor
    )

    if captured_feature_map is None:

        raise RuntimeError(
            "DINO layer4 hook did not capture output."
        )

    if captured_feature_map.ndim != 4:

        raise RuntimeError(
            "Expected [B,C,H,W], got "
            f"{captured_feature_map.shape}"
        )

    batch_size, channels, grid_h, grid_w = (
        captured_feature_map.shape
    )

    patch_features = (
        captured_feature_map[0]
        .permute(
            1,
            2,
            0
        )
        .reshape(
            grid_h * grid_w,
            channels
        )
        .contiguous()
    )

    return (
        patch_features,
        grid_h,
        grid_w
    )


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_features, grid_h, grid_w = (
    extract_dino_patch_features(
        test_image
    )
)

print("=" * 70)
print("DINO PATCH TEST")
print("=" * 70)

print(
    "Raw feature map:",
    captured_feature_map.shape
)

print(
    "Grid:",
    f"{grid_h} × {grid_w}"
)

print(
    "Patch features:",
    test_features.shape
)

assert test_features.shape == (
    49,
    2048
)

print(
    "\n✓ DINO extraction gives [49, 2048]"
)

print("=" * 70)

DINO PATCH TEST
Raw feature map: torch.Size([1, 2048, 7, 7])
Grid: 7 × 7
Patch features: torch.Size([49, 2048])

✓ DINO extraction gives [49, 2048]


In [52]:
# ============================================================
# CELL 13 — LOAD VEGANN AGRICLIP ALIGNER
# ============================================================

print("=" * 70)
print("LOADING VEGANN ALIGNER")
print("=" * 70)

aligner_checkpoint = torch.load(
    ALIGNER_PATH,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint keys:",
    list(aligner_checkpoint.keys())
)

W = torch.as_tensor(
    aligner_checkpoint["W"],
    dtype=torch.float32
)

b = torch.as_tensor(
    aligner_checkpoint["b"],
    dtype=torch.float32
)

print(
    "W:",
    W.shape
)

print(
    "b:",
    b.shape
)

# ------------------------------------------------------------
# Normalize orientation
# ------------------------------------------------------------

if W.shape == (
    512,
    2048
):

    print(
        "Detected W = [512, 2048]"
    )

elif W.shape == (
    2048,
    512
):

    W = W.T

    print(
        "Detected W = [2048, 512]; transposed."
    )

else:

    raise RuntimeError(
        f"Unexpected aligner W shape: {W.shape}"
    )

b = b.reshape(-1)

assert W.shape == (
    512,
    2048
)

assert b.shape == (
    512,
)

vegann_aligner = nn.Linear(
    2048,
    512,
    bias=True
)

with torch.no_grad():

    vegann_aligner.weight.copy_(W)
    vegann_aligner.bias.copy_(b)

vegann_aligner = (
    vegann_aligner
    .to(device)
)

vegann_aligner.eval()

for parameter in (
    vegann_aligner.parameters()
):

    parameter.requires_grad = False

print(
    "\n✓ VegAnn aligner loaded"
)

print(
    "Weight:",
    vegann_aligner.weight.shape
)

print(
    "Bias:",
    vegann_aligner.bias.shape
)

print("=" * 70)

LOADING VEGANN ALIGNER
Checkpoint keys: ['b', 'W']
W: torch.Size([512, 2048])
b: torch.Size([512])
Detected W = [512, 2048]

✓ VegAnn aligner loaded
Weight: torch.Size([512, 2048])
Bias: torch.Size([512])


In [53]:
# ============================================================
# CELL 14 — CREATE TEXTTOCONCEPT
# ============================================================

print("=" * 70)
print("CREATING TEXTTOCONCEPT")
print("=" * 70)

text_to_concept = TextToConcept(
    dino_model,
    "Dino"
)

print(
    "TextToConcept:",
    type(text_to_concept)
)

print(
    "Internal CLIP:",
    type(text_to_concept.clip_model)
)

print(
    "\n✓ TextToConcept created"
)

print("=" * 70)

CREATING TEXTTOCONCEPT
TextToConcept: <class 'TextToConcept.TextToConcept'>
Internal CLIP: <class 'TextToConcept.ClipZeroShot'>

✓ TextToConcept created


In [54]:
# ============================================================
# CELL 15 — LOAD ALIGNER INTO TEXTTOCONCEPT
# ============================================================

text_to_concept.load_linear_aligner(
    str(ALIGNER_PATH)
)

print("=" * 70)
print("TEXTTOCONCEPT ALIGNER")
print("=" * 70)

print(
    "✓ Linear aligner loaded into TextToConcept"
)

print("=" * 70)

TEXTTOCONCEPT ALIGNER
✓ Linear aligner loaded into TextToConcept


In [56]:
# ============================================================
# CELL 16 — VEGETATION PROMPTS
# ============================================================

POSITIVE_PROMPTS = [

    "vegetation",
    "green vegetation",
    "crop plants",
    "crop vegetation",
    "agricultural plants",
    "green plants",
]

NEGATIVE_PROMPTS = [

    "soil",
    "bare soil",
    "ground",
    "bare ground",
    "non-vegetated ground",
    "agricultural soil",
]

print("Prompt strategy: POSITIVE / NEGATIVE")

print("\nPositive prompts:")
for i, prompt in enumerate(POSITIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")

print("\nNegative prompts:")
for i, prompt in enumerate(NEGATIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")
    
    
    


Prompt strategy: POSITIVE / NEGATIVE

Positive prompts:
1. vegetation
2. green vegetation
3. crop plants
4. crop vegetation
5. agricultural plants
6. green plants

Negative prompts:
1. soil
2. bare soil
3. ground
4. bare ground
5. non-vegetated ground
6. agricultural soil


In [57]:

# ============================================================
# CELL 17 ENCODE POSITIVE AND NEGATIVE PROMPT SETS
# ============================================================

@torch.no_grad()
def encode_prompt_set(prompts):

    embeddings = []

    for prompt in prompts:

        embedding = text_to_concept.encode_text(
            [[prompt]]
        )

        if isinstance(embedding, np.ndarray):
            embedding = torch.from_numpy(
                embedding
            )

        if not torch.is_tensor(embedding):
            raise TypeError(
                "encode_text returned an unsupported type: "
                f"{type(embedding)}"
            )

        embedding = (
            embedding
            .detach()
            .float()
            .to(device)
            .reshape(-1)
        )

        embeddings.append(
            embedding
        )

    embeddings = torch.stack(
        embeddings,
        dim=0
    )

    # Normalize each prompt independently
    embeddings = F.normalize(
        embeddings,
        dim=1
    )

    # Average prompts
    mean_embedding = embeddings.mean(
        dim=0
    )

    # Normalize the averaged embedding
    mean_embedding = F.normalize(
        mean_embedding,
        dim=0
    )

    return embeddings, mean_embedding


# ------------------------------------------------------------
# POSITIVE
# ------------------------------------------------------------

positive_text_embeddings, positive_embedding = (
    encode_prompt_set(
        POSITIVE_PROMPTS
    )
)


# ------------------------------------------------------------
# NEGATIVE
# ------------------------------------------------------------

negative_text_embeddings, negative_embedding = (
    encode_prompt_set(
        NEGATIVE_PROMPTS
    )
)


print(
    "Positive embeddings:",
    positive_text_embeddings.shape
)

print(
    "Negative embeddings:",
    negative_text_embeddings.shape
)

print(
    "Positive mean embedding:",
    positive_embedding.shape
)

print(
    "Negative mean embedding:",
    negative_embedding.shape
)

Positive embeddings: torch.Size([6, 512])
Negative embeddings: torch.Size([6, 512])
Positive mean embedding: torch.Size([512])
Negative mean embedding: torch.Size([512])


In [58]:
# ============================================================
# POSITIVE - NEGATIVE CONCEPT
# ============================================================

vegetation_embedding = (
    positive_embedding
    - negative_embedding
)

vegetation_embedding = F.normalize(
    vegetation_embedding,
    dim=0
)

print(
    "Final vegetation embedding:",
    vegetation_embedding.shape
)

Final vegetation embedding: torch.Size([512])


In [79]:
# ============================================================
# CELL — DEFINE NON-EMPTY VEGANN EVALUATION SET
# ============================================================

NONEMPTY_INDICES = []
EMPTY_INDICES = []

for idx in range(len(val_hf)):

    gt_mask = get_mask(
        idx
    )

    if gt_mask.sum() == 0:

        EMPTY_INDICES.append(
            idx
        )

    else:

        NONEMPTY_INDICES.append(
            idx
        )

print("=" * 70)
print("VEGANN EVALUATION SET")
print("=" * 70)

print(
    "Total validation images:",
    len(val_hf)
)

print(
    "Non-empty GT images:",
    len(NONEMPTY_INDICES)
)

print(
    "Empty GT images:",
    len(EMPTY_INDICES)
)

print(
    "Percentage empty:",
    f"{100 * len(EMPTY_INDICES) / len(val_hf):.2f}%"
)

print(
    "\nEmpty image IDs:"
)

print(
    EMPTY_INDICES
)

assert len(EMPTY_INDICES) == 9
assert len(NONEMPTY_INDICES) == 746

print("\n✓ Evaluation sets created")

VEGANN EVALUATION SET
Total validation images: 755
Non-empty GT images: 746
Empty GT images: 9
Percentage empty: 1.19%

Empty image IDs:
[81, 281, 348, 429, 454, 497, 498, 552, 644]

✓ Evaluation sets created


In [59]:
# ============================================================
# CELL 18 — TEST DINO → ALIGNER → TEXT SIMILARITY
# ============================================================

with torch.no_grad():

    aligned_test = (
        vegann_aligner(
            test_features.to(device)
        )
    )

    aligned_test = F.normalize(
        aligned_test,
        dim=-1
    )

    test_scores = (
        aligned_test
        @ vegetation_embedding
    )

print("=" * 70)
print("AGRICLIP SPATIAL SCORING")
print("=" * 70)

print(
    "DINO features:",
    test_features.shape
)

print(
    "Aligned features:",
    aligned_test.shape
)

print(
    "Vegetation embedding:",
    vegetation_embedding.shape
)

print(
    "Patch scores:",
    test_scores.shape
)

print(
    "\nScore min:",
    test_scores.min().item()
)

print(
    "Score max:",
    test_scores.max().item()
)

top_preview = torch.topk(
    test_scores,
    k=10
)

print(
    "\nTop-10 patch indices:",
    top_preview.indices.cpu().numpy()
)

print(
    "Top-10 scores:",
    top_preview.values.cpu().numpy()
)

print("=" * 70)

AGRICLIP SPATIAL SCORING
DINO features: torch.Size([49, 2048])
Aligned features: torch.Size([49, 512])
Vegetation embedding: torch.Size([512])
Patch scores: torch.Size([49])

Score min: -0.020329272374510765
Score max: 0.14176683127880096

Top-10 patch indices: [13 43  6 20 29 10  1 47 18  8]
Top-10 scores: [0.14176683 0.1323612  0.13027082 0.12215064 0.12186894 0.12176584
 0.1201911  0.1149954  0.11485691 0.11447184]


In [60]:
# ============================================================
# CELL 19 — AGRICLIP PATCHES → BOX + POINTS
# ============================================================

def agriclip_patches_to_box(
    image,
    top_indices,
    grid_h,
    grid_w,
    margin=25
):

    W, H = image.size

    patch_h = H / grid_h
    patch_w = W / grid_w

    patch_boxes = []

    for idx in top_indices:

        row = (
            int(idx) //
            grid_w
        )

        col = (
            int(idx) %
            grid_w
        )

        x1 = int(
            col * patch_w
        )

        y1 = int(
            row * patch_h
        )

        x2 = int(
            (col + 1) * patch_w
        )

        y2 = int(
            (row + 1) * patch_h
        )

        patch_boxes.append(
            [
                x1,
                y1,
                x2,
                y2
            ]
        )

    patch_boxes = np.asarray(
        patch_boxes,
        dtype=np.float32
    )

    xmin = patch_boxes[:, 0].min()
    ymin = patch_boxes[:, 1].min()
    xmax = patch_boxes[:, 2].max()
    ymax = patch_boxes[:, 3].max()

    xmin = max(
        0,
        xmin - margin
    )

    ymin = max(
        0,
        ymin - margin
    )

    xmax = min(
        W - 1,
        xmax + margin
    )

    ymax = min(
        H - 1,
        ymax + margin
    )

    return np.array(
        [
            xmin,
            ymin,
            xmax,
            ymax
        ],
        dtype=np.float32
    )


def agriclip_patches_to_points(
    image,
    top_indices,
    grid_h,
    grid_w
):

    W, H = image.size

    patch_h = H / grid_h
    patch_w = W / grid_w

    points = []

    for idx in top_indices:

        row = (
            int(idx) //
            grid_w
        )

        col = (
            int(idx) %
            grid_w
        )

        x1 = col * patch_w
        y1 = row * patch_h

        x2 = (
            col + 1
        ) * patch_w

        y2 = (
            row + 1
        ) * patch_h

        points.append(
            [
                (x1 + x2) / 2,
                (y1 + y2) / 2
            ]
        )

    return np.asarray(
        points,
        dtype=np.float32
    )


print(
    "✓ Box and point conversion functions ready"
)

✓ Box and point conversion functions ready


In [61]:
# ============================================================
# CELL 20 — LOAD ORIGINAL FINE-TUNED SAM2
# ============================================================

from transformers import (
    Sam2Model,
    Sam2Processor
)

print("=" * 70)
print("LOADING ORIGINAL FINE-TUNED SAM2")
print("=" * 70)

sam2_processor = (
    Sam2Processor.from_pretrained(
        MODEL_NAME
    )
)

sam2_model = (
    Sam2Model.from_pretrained(
        MODEL_NAME
    )
)

sam2_checkpoint = torch.load(
    SAM2_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint keys:",
    list(sam2_checkpoint.keys())
)

if (
    "mask_decoder_state_dict"
    not in sam2_checkpoint
):

    raise RuntimeError(
        "Expected mask_decoder_state_dict "
        "in SAM2 checkpoint."
    )

sam2_model.mask_decoder.load_state_dict(
    sam2_checkpoint[
        "mask_decoder_state_dict"
    ],
    strict=True
)

sam2_model = (
    sam2_model.to(device)
)

sam2_model.eval()

print(
    "\n✓ Fine-tuned non-augmented SAM2 loaded"
)

print(
    "Checkpoint epoch:",
    sam2_checkpoint.get("epoch")
)

print(
    "Best stored IoU:",
    sam2_checkpoint.get(
        "best_mean_iou"
    )
)

print("=" * 70)

LOADING ORIGINAL FINE-TUNED SAM2


[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Checkpoint keys: ['epoch', 'model_name', 'mask_decoder_state_dict', 'optimizer_state_dict', 'best_mean_iou', 'history']

✓ Fine-tuned non-augmented SAM2 loaded
Checkpoint epoch: 8
Best stored IoU: 0.8724805513351916


In [62]:
# ============================================================
# CELL 21 — SAM2 BOX PREDICTION
# ============================================================

@torch.no_grad()
def sam2_box_prediction(
    image,
    box,
    threshold=0.5
):

    inputs = sam2_processor(
        images=image,
        input_boxes=[
            [
                box.tolist()
            ]
        ],
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    outputs = sam2_model(
        **inputs
    )

    logits = outputs.pred_masks[
        0, 0
    ]

    scores = outputs.iou_scores[
        0, 0
    ]

    probabilities = torch.sigmoid(
        logits
    )

    height = int(
        inputs[
            "original_sizes"
        ][0, 0].item()
    )

    width = int(
        inputs[
            "original_sizes"
        ][0, 1].item()
    )

    probabilities = F.interpolate(
        probabilities.unsqueeze(1),
        size=(
            height,
            width
        ),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    candidate = (
        scores.argmax().item()
    )

    prediction = (
        probabilities[candidate]
        > threshold
    ).cpu().numpy().astype(
        np.uint8
    )

    return {
        "prediction": prediction,
        "confidence": float(
            scores[candidate].item()
        ),
        "candidate": candidate,
        "probabilities": probabilities.cpu(),
        "scores": scores.cpu()
    }


print(
    "✓ SAM2 box prediction ready"
)

✓ SAM2 box prediction ready


In [63]:
# ============================================================
# CELL 22 — SAM2 POINT PREDICTION
# ============================================================

@torch.no_grad()
def sam2_point_prediction(
    image,
    points,
    threshold=0.5
):

    points = np.asarray(
        points,
        dtype=np.float32
    )

    if points.ndim != 2:
        raise ValueError(
            f"Expected [N,2], "
            f"got {points.shape}"
        )

    input_points = [
        [
            points.tolist()
        ]
    ]

    input_labels = [
        [
            [1] * len(points)
        ]
    ]

    inputs = sam2_processor(
        images=image,
        input_points=input_points,
        input_labels=input_labels,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    outputs = sam2_model(
        **inputs
    )

    logits = outputs.pred_masks[
        0, 0
    ]

    scores = outputs.iou_scores[
        0, 0
    ]

    probabilities = torch.sigmoid(
        logits
    )

    height = int(
        inputs[
            "original_sizes"
        ][0, 0].item()
    )

    width = int(
        inputs[
            "original_sizes"
        ][0, 1].item()
    )

    probabilities = F.interpolate(
        probabilities.unsqueeze(1),
        size=(
            height,
            width
        ),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    candidate = (
        scores.argmax().item()
    )

    prediction = (
        probabilities[candidate]
        > threshold
    ).cpu().numpy().astype(
        np.uint8
    )

    return {
        "prediction": prediction,
        "confidence": float(
            scores[candidate].item()
        ),
        "candidate": candidate,
        "probabilities":
            probabilities.cpu(),
        "scores":
            scores.cpu()
    }


print(
    "✓ SAM2 point prediction ready"
)

✓ SAM2 point prediction ready


In [64]:
# ============================================================
# CELL 23 — SAM2 BOX + POSITIVE POINTS
# ============================================================

@torch.no_grad()
def sam2_box_point_prediction(
    image,
    box,
    points,
    threshold=0.5
):

    points = np.asarray(
        points,
        dtype=np.float32
    )

    if points.ndim != 2:
        raise ValueError(
            f"Expected points [N,2], "
            f"got {points.shape}"
        )

    input_boxes = [
        [
            box.tolist()
        ]
    ]

    input_points = [
        [
            points.tolist()
        ]
    ]

    input_labels = [
        [
            [1] * len(points)
        ]
    ]

    inputs = sam2_processor(
        images=image,
        input_boxes=input_boxes,
        input_points=input_points,
        input_labels=input_labels,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    outputs = sam2_model(
        **inputs
    )

    logits = outputs.pred_masks[
        0, 0
    ]

    scores = outputs.iou_scores[
        0, 0
    ]

    probabilities = torch.sigmoid(
        logits
    )

    height = int(
        inputs[
            "original_sizes"
        ][0, 0].item()
    )

    width = int(
        inputs[
            "original_sizes"
        ][0, 1].item()
    )

    probabilities = F.interpolate(
        probabilities.unsqueeze(1),
        size=(
            height,
            width
        ),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    candidate = (
        scores.argmax().item()
    )

    prediction = (
        probabilities[candidate]
        > threshold
    ).cpu().numpy().astype(
        np.uint8
    )

    return {
        "prediction": prediction,
        "confidence": float(
            scores[candidate].item()
        ),
        "candidate": candidate,
        "probabilities":
            probabilities.cpu(),
        "scores":
            scores.cpu()
    }


print(
    "✓ SAM2 box + points prediction ready"
)

✓ SAM2 box + points prediction ready


In [80]:
# ============================================================
# CELL 24 — METRICS
# ============================================================

# ============================================================
# CELL — ROBUST SEGMENTATION METRICS
# ============================================================

def compute_metrics_robust(
    prediction,
    ground_truth
):
    """
    Compute segmentation metrics.

    For the main VegAnn segmentation evaluation, empty
    ground-truth images should normally be excluded before
    calculating the dataset mean.

    This function also handles the empty/empty case safely.
    """

    prediction = (
        prediction > 0
    ).astype(bool)

    ground_truth = (
        ground_truth > 0
    ).astype(bool)

    pred_area = prediction.sum()
    gt_area = ground_truth.sum()

    intersection = np.logical_and(
        prediction,
        ground_truth
    ).sum()

    union = np.logical_or(
        prediction,
        ground_truth
    ).sum()

    # --------------------------------------------------------
    # Empty GT
    # --------------------------------------------------------

    if gt_area == 0:

        # Completely empty prediction
        if pred_area == 0:

            return {
                "iou": 1.0,
                "dice": 1.0,
                "precision": 1.0,
                "recall": 1.0,
                "empty_gt": True,
                "empty_prediction": True
            }

        # False positive prediction on empty image
        else:

            return {
                "iou": 0.0,
                "dice": 0.0,
                "precision": 0.0,
                "recall": 0.0,
                "empty_gt": True,
                "empty_prediction": False
            }

    # --------------------------------------------------------
    # Normal non-empty GT case
    # --------------------------------------------------------

    iou = (
        intersection / union
        if union > 0
        else 0.0
    )

    dice_den = (
        pred_area +
        gt_area
    )

    dice = (
        2.0 * intersection / dice_den
        if dice_den > 0
        else 0.0
    )

    precision = (
        intersection / pred_area
        if pred_area > 0
        else 0.0
    )

    recall = (
        intersection / gt_area
    )

    return {
        "iou": float(iou),
        "dice": float(dice),
        "precision": float(precision),
        "recall": float(recall),
        "empty_gt": False,
        "empty_prediction": (
            pred_area == 0
        )
    }


print("✓ Robust metric function ready.")

✓ Robust metric function ready.


In [66]:
# ============================================================
# CELL 25 — GT MASK → GT BOX
# ============================================================

def mask_to_box(
    mask
):

    ys, xs = np.where(
        mask > 0
    )

    if len(xs) == 0:

        raise ValueError(
            "Ground-truth mask is empty."
        )

    return np.array(
        [
            xs.min(),
            ys.min(),
            xs.max(),
            ys.max()
        ],
        dtype=np.float32
    )


def box_iou(
    box_a,
    box_b
):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(
        ax1,
        bx1
    )

    iy1 = max(
        ay1,
        by1
    )

    ix2 = min(
        ax2,
        bx2
    )

    iy2 = min(
        ay2,
        by2
    )

    iw = max(
        0,
        ix2 - ix1
    )

    ih = max(
        0,
        iy2 - iy1
    )

    intersection = (
        iw * ih
    )

    area_a = (
        max(
            0,
            ax2 - ax1
        )
        *
        max(
            0,
            ay2 - ay1
        )
    )

    area_b = (
        max(
            0,
            bx2 - bx1
        )
        *
        max(
            0,
            by2 - by1
        )
    )

    union = (
        area_a +
        area_b -
        intersection
    )

    if union == 0:
        return 0.0

    return (
        intersection /
        union
    )


print(
    "✓ GT box helpers ready"
)

✓ GT box helpers ready


In [67]:
# ============================================================
# CELL 26 — SINGLE IMAGE COMPLETE TEST
# ============================================================

TEST_IDX = 0
K = 32

image = get_image(
    TEST_IDX
)

gt_mask = get_mask(
    TEST_IDX
)

# ------------------------------------------------------------
# AgriCLIP
# ------------------------------------------------------------

loc = get_agriclip_localization(
    image,
    k=K
)

# ------------------------------------------------------------
# Create box
# ------------------------------------------------------------

agriclip_box = (
    agriclip_patches_to_box(
        image,
        loc["top_indices"],
        loc["grid_h"],
        loc["grid_w"],
        margin=25
    )
)

# ------------------------------------------------------------
# Create points
# ------------------------------------------------------------

agriclip_points = (
    agriclip_patches_to_points(
        image,
        loc["top_indices"],
        loc["grid_h"],
        loc["grid_w"]
    )
)

# ------------------------------------------------------------
# GT box
# ------------------------------------------------------------

gt_box = mask_to_box(
    gt_mask
)

# ------------------------------------------------------------
# SAM2 box
# ------------------------------------------------------------

box_result = sam2_box_prediction(
    image,
    agriclip_box
)

box_metrics = compute_metrics(
    box_result["prediction"],
    gt_mask
)

# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("=" * 70)
print("SINGLE IMAGE COMPLETE TEST")
print("=" * 70)

print(
    "Image:",
    TEST_IDX
)

print(
    "K:",
    K
)

print(
    "\nAgriCLIP box:",
    agriclip_box
)

print(
    "\nGT box:",
    gt_box
)

print(
    "\nBox IoU:",
    f"{100 * box_iou(agriclip_box, gt_box):.2f}%"
)

print(
    "\nSAM2 candidate:",
    box_result["candidate"]
)

print(
    "SAM2 confidence:",
    f"{box_result['confidence']:.4f}"
)

print(
    "SAM2 mask IoU:",
    f"{100 * box_metrics['iou']:.2f}%"
)

print(
    "SAM2 Dice:",
    f"{100 * box_metrics['dice']:.2f}%"
)

print(
    "SAM2 precision:",
    f"{100 * box_metrics['precision']:.2f}%"
)

print(
    "SAM2 recall:",
    f"{100 * box_metrics['recall']:.2f}%"
)

print("=" * 70)

SINGLE IMAGE COMPLETE TEST
Image: 0
K: 32

AgriCLIP box: [  0.   0. 511. 511.]

GT box: [  0.   0. 511. 511.]

Box IoU: 100.00%

SAM2 candidate: 2
SAM2 confidence: 0.9524
SAM2 mask IoU: 90.22%
SAM2 Dice: 94.86%
SAM2 precision: 94.79%
SAM2 recall: 94.92%


In [68]:
# ============================================================
# CELL 27 — SINGLE IMAGE POINT PROMPT TEST
# ============================================================

POINT_COUNTS = [
    1,
    3,
    5,
    8
]

print("=" * 70)
print("AGRICLIP POINT PROMPT TEST")
print("=" * 70)

for n in POINT_COUNTS:

    points = (
        agriclip_points[:n]
    )

    result = sam2_point_prediction(
        image,
        points
    )

    metrics = compute_metrics(
        result["prediction"],
        gt_mask
    )

    print(
        f"\nTop-{n} AgriCLIP points"
    )

    print(
        f"IoU: "
        f"{100 * metrics['iou']:.2f}%"
    )

    print(
        f"Dice: "
        f"{100 * metrics['dice']:.2f}%"
    )

    print(
        f"Confidence: "
        f"{result['confidence']:.4f}"
    )

print("=" * 70)

AGRICLIP POINT PROMPT TEST

Top-1 AgriCLIP points
IoU: 87.91%
Dice: 93.56%
Confidence: 0.9224

Top-3 AgriCLIP points
IoU: 81.88%
Dice: 90.04%
Confidence: 0.9329

Top-5 AgriCLIP points
IoU: 80.84%
Dice: 89.41%
Confidence: 0.9231

Top-8 AgriCLIP points
IoU: 78.46%
Dice: 87.93%
Confidence: 0.9234


In [69]:
# ============================================================
# CELL 28 — SINGLE IMAGE BOX + POINT TEST
# ============================================================

print("=" * 70)
print("AGRICLIP BOX + POINT PROMPT TEST")
print("=" * 70)

for n in POINT_COUNTS:

    points = (
        agriclip_points[:n]
    )

    result = (
        sam2_box_point_prediction(
            image,
            agriclip_box,
            points
        )
    )

    metrics = compute_metrics(
        result["prediction"],
        gt_mask
    )

    print(
        f"\nBox + top-{n} AgriCLIP points"
    )

    print(
        f"IoU: "
        f"{100 * metrics['iou']:.2f}%"
    )

    print(
        f"Dice: "
        f"{100 * metrics['dice']:.2f}%"
    )

    print(
        f"Confidence: "
        f"{result['confidence']:.4f}"
    )

print("=" * 70)

AGRICLIP BOX + POINT PROMPT TEST

Box + top-1 AgriCLIP points
IoU: 89.46%
Dice: 94.44%
Confidence: 0.9525

Box + top-3 AgriCLIP points
IoU: 88.37%
Dice: 93.83%
Confidence: 0.9494

Box + top-5 AgriCLIP points
IoU: 86.57%
Dice: 92.80%
Confidence: 0.9500

Box + top-8 AgriCLIP points
IoU: 85.51%
Dice: 92.19%
Confidence: 0.9458


In [71]:
# ============================================================
# CELL 29 — FULL GT BOX → SAM2
# ROBUST VERSION: HANDLES EMPTY GT MASKS
# ============================================================

gt_results = []

empty_gt_images = []

print("=" * 70)
print("GT BOX → SAM2")
print("=" * 70)

for idx in tqdm(
    range(len(val_hf)),
    desc="GT Box -> SAM2"
):

    image = get_image(
        idx
    )

    gt_mask = get_mask(
        idx
    )

    # --------------------------------------------------------
    # Check for empty ground-truth mask
    # --------------------------------------------------------

    if gt_mask.sum() == 0:

        empty_gt_images.append(
            idx
        )

        continue

    # --------------------------------------------------------
    # Create GT bounding box
    # --------------------------------------------------------

    gt_box = mask_to_box(
        gt_mask
    )

    # --------------------------------------------------------
    # SAM2 prediction
    # --------------------------------------------------------

    result = sam2_box_prediction(
        image,
        gt_box
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = compute_metrics(
        result["prediction"],
        gt_mask
    )

    gt_results.append(
        {
            "image_id": idx,
            "iou": metrics["iou"],
            "dice": metrics["dice"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "confidence": result["confidence"]
        }
    )


# ============================================================
# DATAFRAME
# ============================================================

gt_df = pd.DataFrame(
    gt_results
)


# ============================================================
# RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("GT BOX → SAM2 RESULTS")
print("=" * 70)

print(
    "Total validation images:",
    len(val_hf)
)

print(
    "Images evaluated:",
    len(gt_df)
)

print(
    "Empty-GT images skipped:",
    len(empty_gt_images)
)

if len(empty_gt_images) > 0:

    print(
        "\nEmpty-GT image IDs:"
    )

    print(
        empty_gt_images
    )

if len(gt_df) > 0:

    print(
        f"\nMean IoU: "
        f"{100 * gt_df['iou'].mean():.2f}%"
    )

    print(
        f"Mean Dice: "
        f"{100 * gt_df['dice'].mean():.2f}%"
    )

    print(
        f"Mean Precision: "
        f"{100 * gt_df['precision'].mean():.2f}%"
    )

    print(
        f"Mean Recall: "
        f"{100 * gt_df['recall'].mean():.2f}%"
    )

else:

    print(
        "\nNo non-empty GT masks were found."
    )

print("=" * 70)

GT BOX → SAM2


GT Box -> SAM2:   0%|          | 0/755 [00:00<?, ?it/s]



GT BOX → SAM2 RESULTS
Total validation images: 755
Images evaluated: 746
Empty-GT images skipped: 9

Empty-GT image IDs:
[81, 281, 348, 429, 454, 497, 498, 552, 644]

Mean IoU: 85.68%
Mean Dice: 91.75%
Mean Precision: 90.96%
Mean Recall: 93.30%


In [83]:
# ============================================================
# CELL 30 — AGRICLIP BOX → SAM2
# 746 NON-EMPTY VEGANN IMAGES
# ============================================================

agriclip_box_results = []

K = 32

print("=" * 70)
print("AGRICLIP BOX → SAM2")
print("NON-EMPTY GT IMAGES ONLY")
print("K =", K)
print("=" * 70)

for idx in tqdm(
    NONEMPTY_INDICES,
    desc="AgriCLIP Box -> SAM2"
):

    image = get_image(idx)
    gt_mask = get_mask(idx)

    # --------------------------------------------------------
    # AgriCLIP localization
    # --------------------------------------------------------

    loc = get_agriclip_localization(
        image,
        k=K
    )

    # --------------------------------------------------------
    # AgriCLIP merged box
    # --------------------------------------------------------

    box = agriclip_patches_to_box(
        image,
        loc["top_indices"],
        loc["grid_h"],
        loc["grid_w"],
        margin=25
    )

    # --------------------------------------------------------
    # SAM2
    # --------------------------------------------------------

    result = sam2_box_prediction(
        image,
        box
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = compute_metrics_robust(
        result["prediction"],
        gt_mask
    )

    agriclip_box_results.append({
        "image_id": idx,
        "iou": metrics["iou"],
        "dice": metrics["dice"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "confidence": result["confidence"]
    })


agriclip_box_df = pd.DataFrame(
    agriclip_box_results
)

print("\n")
print("=" * 70)
print("AGRICLIP BOX → SAM2 RESULTS")
print("=" * 70)

print(
    "Total validation images:",
    len(val_hf)
)

print(
    "Non-empty images evaluated:",
    len(agriclip_box_df)
)

print(
    "Empty images excluded:",
    len(EMPTY_INDICES)
)

print(
    f"\nMean IoU: "
    f"{100 * agriclip_box_df['iou'].mean():.2f}%"
)

print(
    f"Mean Dice: "
    f"{100 * agriclip_box_df['dice'].mean():.2f}%"
)

print(
    f"Mean Precision: "
    f"{100 * agriclip_box_df['precision'].mean():.2f}%"
)

print(
    f"Mean Recall: "
    f"{100 * agriclip_box_df['recall'].mean():.2f}%"
)

print("=" * 70)

AGRICLIP BOX → SAM2
NON-EMPTY GT IMAGES ONLY
K = 32


AgriCLIP Box -> SAM2:   0%|          | 0/746 [00:00<?, ?it/s]



AGRICLIP BOX → SAM2 RESULTS
Total validation images: 755
Non-empty images evaluated: 746
Empty images excluded: 9

Mean IoU: 85.04%
Mean Dice: 91.17%
Mean Precision: 90.09%
Mean Recall: 93.41%


In [84]:
# ============================================================
# CELL 31 — AGRICLIP POINTS → SAM2
# 746 NON-EMPTY VEGANN IMAGES
# ============================================================

POINT_COUNTS = [
    1,
    3,
    5,
    8
]

agriclip_point_dfs = {}

for num_points in POINT_COUNTS:

    results = []

    print("\n")
    print("=" * 70)
    print(
        f"AGRICLIP TOP-{num_points} POINTS → SAM2"
    )
    print("=" * 70)

    for idx in tqdm(
        NONEMPTY_INDICES,
        desc=f"Points {num_points}"
    ):

        image = get_image(idx)
        gt_mask = get_mask(idx)

        # ----------------------------------------------------
        # AgriCLIP localization
        # Always generate K=32 patches
        # ----------------------------------------------------

        loc = get_agriclip_localization(
            image,
            k=K
        )

        # ----------------------------------------------------
        # Convert selected patches to points
        # ----------------------------------------------------

        all_points = (
            agriclip_patches_to_points(
                image,
                loc["top_indices"],
                loc["grid_h"],
                loc["grid_w"]
            )
        )

        points = all_points[
            :num_points
        ]

        # ----------------------------------------------------
        # SAM2
        # ----------------------------------------------------

        result = sam2_point_prediction(
            image,
            points
        )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        metrics = compute_metrics_robust(
            result["prediction"],
            gt_mask
        )

        results.append({
            "image_id": idx,
            "iou": metrics["iou"],
            "dice": metrics["dice"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "confidence": result["confidence"]
        })

    df = pd.DataFrame(results)

    agriclip_point_dfs[
        num_points
    ] = df

    print(
        f"\nTop-{num_points} points:"
    )

    print(
        f"IoU: "
        f"{100 * df['iou'].mean():.2f}%"
    )

    print(
        f"Dice: "
        f"{100 * df['dice'].mean():.2f}%"
    )

    print(
        f"Precision: "
        f"{100 * df['precision'].mean():.2f}%"
    )

    print(
        f"Recall: "
        f"{100 * df['recall'].mean():.2f}%"
    )



AGRICLIP TOP-1 POINTS → SAM2


Points 1:   0%|          | 0/746 [00:00<?, ?it/s]


Top-1 points:
IoU: 80.45%
Dice: 87.82%
Precision: 92.19%
Recall: 85.71%


AGRICLIP TOP-3 POINTS → SAM2


Points 3:   0%|          | 0/746 [00:00<?, ?it/s]


Top-3 points:
IoU: 76.41%
Dice: 84.84%
Precision: 92.62%
Recall: 81.01%


AGRICLIP TOP-5 POINTS → SAM2


Points 5:   0%|          | 0/746 [00:00<?, ?it/s]


Top-5 points:
IoU: 75.58%
Dice: 84.14%
Precision: 92.47%
Recall: 80.17%


AGRICLIP TOP-8 POINTS → SAM2


Points 8:   0%|          | 0/746 [00:00<?, ?it/s]


Top-8 points:
IoU: 74.94%
Dice: 83.78%
Precision: 92.86%
Recall: 79.11%


In [85]:
# ============================================================
# CELL 32 — AGRICLIP BOX + POINTS → SAM2
# 746 NON-EMPTY VEGANN IMAGES
# ============================================================

agriclip_box_point_dfs = {}

for num_points in POINT_COUNTS:

    results = []

    print("\n")
    print("=" * 70)
    print(
        f"AGRICLIP BOX + TOP-{num_points} POINTS → SAM2"
    )
    print("=" * 70)

    for idx in tqdm(
        NONEMPTY_INDICES,
        desc=f"Box + {num_points} points"
    ):

        image = get_image(idx)
        gt_mask = get_mask(idx)

        # ----------------------------------------------------
        # AgriCLIP localization
        # ----------------------------------------------------

        loc = get_agriclip_localization(
            image,
            k=K
        )

        # ----------------------------------------------------
        # Box
        # ----------------------------------------------------

        box = agriclip_patches_to_box(
            image,
            loc["top_indices"],
            loc["grid_h"],
            loc["grid_w"],
            margin=25
        )

        # ----------------------------------------------------
        # Points
        # ----------------------------------------------------

        all_points = (
            agriclip_patches_to_points(
                image,
                loc["top_indices"],
                loc["grid_h"],
                loc["grid_w"]
            )
        )

        points = all_points[
            :num_points
        ]

        # ----------------------------------------------------
        # SAM2
        # ----------------------------------------------------

        result = sam2_box_point_prediction(
            image,
            box,
            points
        )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        metrics = compute_metrics_robust(
            result["prediction"],
            gt_mask
        )

        results.append({
            "image_id": idx,
            "iou": metrics["iou"],
            "dice": metrics["dice"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "confidence": result["confidence"]
        })

    df = pd.DataFrame(results)

    agriclip_box_point_dfs[
        num_points
    ] = df

    print(
        f"\nBox + top-{num_points} points:"
    )

    print(
        f"IoU: "
        f"{100 * df['iou'].mean():.2f}%"
    )

    print(
        f"Dice: "
        f"{100 * df['dice'].mean():.2f}%"
    )

    print(
        f"Precision: "
        f"{100 * df['precision'].mean():.2f}%"
    )

    print(
        f"Recall: "
        f"{100 * df['recall'].mean():.2f}%"
    )



AGRICLIP BOX + TOP-1 POINTS → SAM2


Box + 1 points:   0%|          | 0/746 [00:00<?, ?it/s]


Box + top-1 points:
IoU: 84.77%
Dice: 91.05%
Precision: 90.48%
Recall: 92.59%


AGRICLIP BOX + TOP-3 POINTS → SAM2


Box + 3 points:   0%|          | 0/746 [00:00<?, ?it/s]


Box + top-3 points:
IoU: 83.69%
Dice: 90.30%
Precision: 90.97%
Recall: 90.86%


AGRICLIP BOX + TOP-5 POINTS → SAM2


Box + 5 points:   0%|          | 0/746 [00:00<?, ?it/s]


Box + top-5 points:
IoU: 82.81%
Dice: 89.64%
Precision: 91.08%
Recall: 89.74%


AGRICLIP BOX + TOP-8 POINTS → SAM2


Box + 8 points:   0%|          | 0/746 [00:00<?, ?it/s]


Box + top-8 points:
IoU: 82.01%
Dice: 89.11%
Precision: 91.63%
Recall: 88.26%


In [87]:
# ============================================================
# CELL 33 — SAM2 AUTOMATIC GRID-POINT BASELINE
# ============================================================

@torch.no_grad()
def sam2_automatic_grid_prediction(
    image,
    grid_size=4,
    threshold=0.5
):

    W, H = image.size

    xs = np.linspace(
        0.1 * W,
        0.9 * W,
        grid_size
    )

    ys = np.linspace(
        0.1 * H,
        0.9 * H,
        grid_size
    )

    points = []

    for y in ys:

        for x in xs:

            points.append(
                [
                    float(x),
                    float(y)
                ]
            )

    candidate_masks = []
    candidate_scores = []

    for point in points:

        result = sam2_point_prediction(
            image,
            np.asarray(
                [point],
                dtype=np.float32
            ),
            threshold=threshold
        )

        candidate_masks.append(
            result["prediction"]
        )

        candidate_scores.append(
            result["confidence"]
        )

    # --------------------------------------------------------
    # Deterministic baseline:
    # choose the largest predicted foreground mask
    # --------------------------------------------------------

    areas = [
        mask.sum()
        for mask in candidate_masks
    ]

    selected = int(
        np.argmax(areas)
    )

    return {
        "prediction":
            candidate_masks[selected],
        "confidence":
            candidate_scores[selected],
        "selected_point":
            points[selected]
    }


print(
    "✓ Automatic grid-point baseline ready"
)

✓ Automatic grid-point baseline ready


In [88]:
# ============================================================
# — SAM2 AUTOMATIC PROMPTING
# 746 NON-EMPTY VEGANN IMAGES
# ============================================================

automatic_results = []

print("=" * 70)
print("SAM2 AUTOMATIC PROMPTING")
print("NON-EMPTY GT IMAGES ONLY")
print("=" * 70)

for idx in tqdm(
    NONEMPTY_INDICES,
    desc="SAM2 automatic prompting"
):

    image = get_image(idx)
    gt_mask = get_mask(idx)

    result = sam2_automatic_grid_prediction(
        image,
        grid_size=4,
        threshold=0.5
    )

    metrics = compute_metrics_robust(
        result["prediction"],
        gt_mask
    )

    automatic_results.append({
        "image_id": idx,
        "iou": metrics["iou"],
        "dice": metrics["dice"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "confidence": result["confidence"]
    })


automatic_df = pd.DataFrame(
    automatic_results
)

print("\n")
print("=" * 70)
print("SAM2 AUTOMATIC PROMPTING RESULTS")
print("=" * 70)

print(
    "Images:",
    len(automatic_df)
)

print(
    "Empty images excluded:",
    len(EMPTY_INDICES)
)

print(
    f"\nMean IoU: "
    f"{100 * automatic_df['iou'].mean():.2f}%"
)

print(
    f"Mean Dice: "
    f"{100 * automatic_df['dice'].mean():.2f}%"
)

print(
    f"Mean Precision: "
    f"{100 * automatic_df['precision'].mean():.2f}%"
)

print(
    f"Mean Recall: "
    f"{100 * automatic_df['recall'].mean():.2f}%"
)

print("=" * 70)

SAM2 AUTOMATIC PROMPTING
NON-EMPTY GT IMAGES ONLY


SAM2 automatic prompting:   0%|          | 0/746 [00:00<?, ?it/s]



SAM2 AUTOMATIC PROMPTING RESULTS
Images: 746
Empty images excluded: 9

Mean IoU: 84.13%
Mean Dice: 90.35%
Mean Precision: 89.69%
Mean Recall: 92.05%


In [89]:
# ============================================================
# CELL 34 — FINAL FAIR PROMPT COMPARISON
# ALL METHODS: SAME 746 NON-EMPTY IMAGES
# ============================================================

comparison_rows = []

# ------------------------------------------------------------
# GT Box
# ------------------------------------------------------------

comparison_rows.append({
    "Method":
        "SAM2 + GT Box",
    "Prompt":
        "Ground-truth box",
    "Images":
        len(gt_df),
    "IoU":
        gt_df["iou"].mean(),
    "Dice":
        gt_df["dice"].mean(),
    "Precision":
        gt_df["precision"].mean(),
    "Recall":
        gt_df["recall"].mean()
})

# ------------------------------------------------------------
# SAM2 automatic
# ------------------------------------------------------------

comparison_rows.append({
    "Method":
        "SAM2 Automatic Prompting",
    "Prompt":
        "Automatic grid points",
    "Images":
        len(automatic_df),
    "IoU":
        automatic_df["iou"].mean(),
    "Dice":
        automatic_df["dice"].mean(),
    "Precision":
        automatic_df["precision"].mean(),
    "Recall":
        automatic_df["recall"].mean()
})

# ------------------------------------------------------------
# AgriCLIP box
# ------------------------------------------------------------

comparison_rows.append({
    "Method":
        "AgriCLIP + SAM2",
    "Prompt":
        "AgriCLIP K=32 box",
    "Images":
        len(agriclip_box_df),
    "IoU":
        agriclip_box_df["iou"].mean(),
    "Dice":
        agriclip_box_df["dice"].mean(),
    "Precision":
        agriclip_box_df["precision"].mean(),
    "Recall":
        agriclip_box_df["recall"].mean()
})

# ------------------------------------------------------------
# AgriCLIP points
# ------------------------------------------------------------

for n in POINT_COUNTS:

    df = agriclip_point_dfs[n]

    comparison_rows.append({
        "Method":
            f"AgriCLIP + SAM2 ({n} points)",
        "Prompt":
            f"Top-{n} AgriCLIP points",
        "Images":
            len(df),
        "IoU":
            df["iou"].mean(),
        "Dice":
            df["dice"].mean(),
        "Precision":
            df["precision"].mean(),
        "Recall":
            df["recall"].mean()
    })

# ------------------------------------------------------------
# AgriCLIP box + points
# ------------------------------------------------------------

for n in POINT_COUNTS:

    df = agriclip_box_point_dfs[n]

    comparison_rows.append({
        "Method":
            f"AgriCLIP Box + {n} Points",
        "Prompt":
            f"K=32 box + top-{n} points",
        "Images":
            len(df),
        "IoU":
            df["iou"].mean(),
        "Dice":
            df["dice"].mean(),
        "Precision":
            df["precision"].mean(),
        "Recall":
            df["recall"].mean()
    })


fair_comparison_df = pd.DataFrame(
    comparison_rows
)

print("=" * 70)
print("FAIR PROMPT STRATEGY COMPARISON")
print("746 NON-EMPTY VEGANN VALIDATION IMAGES")
print("=" * 70)

display(
    fair_comparison_df.style.format({
        "IoU": "{:.4f}",
        "Dice": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}"
    })
)

FAIR PROMPT STRATEGY COMPARISON
746 NON-EMPTY VEGANN VALIDATION IMAGES


,Method,Prompt,Images,IoU,Dice,Precision,Recall
0,SAM2 + GT Box,Ground-truth box,746,0.8568,0.9175,0.9096,0.9330
1,SAM2 Automatic Prompting,Automatic grid points,746,0.8413,0.9035,0.8969,0.9205
2,AgriCLIP + SAM2,AgriCLIP K=32 box,746,0.8504,0.9117,0.9009,0.9341
3,AgriCLIP + SAM2 (1 points),Top-1 AgriCLIP points,746,0.8045,0.8782,0.9219,0.8571
4,AgriCLIP + SAM2 (3 points),Top-3 AgriCLIP points,746,0.7641,0.8484,0.9262,0.8101
5,AgriCLIP + SAM2 (5 points),Top-5 AgriCLIP points,746,0.7558,0.8414,0.9247,0.8017
6,AgriCLIP + SAM2 (8 points),Top-8 AgriCLIP points,746,0.7494,0.8378,0.9286,0.7911
7,AgriCLIP Box + 1 Points,K=32 box + top-1 points,746,0.8477,0.9105,0.9048,0.9259
8,AgriCLIP Box + 3 Points,K=32 box + top-3 points,746,0.8369,0.9030,0.9097,0.9086
9,AgriCLIP Box + 5 Points,K=32 box + top-5 points,746,0.8281,0.8964,0.9108,0.8974


In [90]:
# ============================================================
# CELL 35 — FINAL PROMPT STRATEGY TABLE
# ============================================================

comparison_rows = []

# ------------------------------------------------------------
# GT Box
# ------------------------------------------------------------

comparison_rows.append(
    {
        "Method":
            "SAM2 + GT Box",
        "Prompt":
            "Ground-truth box",
        "IoU":
            gt_df["iou"].mean(),
        "Dice":
            gt_df["dice"].mean(),
        "Precision":
            gt_df["precision"].mean(),
        "Recall":
            gt_df["recall"].mean()
    }
)

# ------------------------------------------------------------
# SAM2 automatic
# ------------------------------------------------------------

comparison_rows.append(
    {
        "Method":
            "SAM2 Automatic Prompting",
        "Prompt":
            "Automatic grid points",
        "IoU":
            automatic_df["iou"].mean(),
        "Dice":
            automatic_df["dice"].mean(),
        "Precision":
            automatic_df["precision"].mean(),
        "Recall":
            automatic_df["recall"].mean()
    }
)

# ------------------------------------------------------------
# AgriCLIP box
# ------------------------------------------------------------

comparison_rows.append(
    {
        "Method":
            "AgriCLIP + SAM2",
        "Prompt":
            "AgriCLIP K=32 box",
        "IoU":
            agriclip_box_df["iou"].mean(),
        "Dice":
            agriclip_box_df["dice"].mean(),
        "Precision":
            agriclip_box_df["precision"].mean(),
        "Recall":
            agriclip_box_df["recall"].mean()
    }
)

# ------------------------------------------------------------
# AgriCLIP points
# ------------------------------------------------------------

for n in POINT_COUNTS:

    df = agriclip_point_dfs[n]

    comparison_rows.append(
        {
            "Method":
                f"AgriCLIP + SAM2 ({n} points)",
            "Prompt":
                f"Top-{n} AgriCLIP points",
            "IoU":
                df["iou"].mean(),
            "Dice":
                df["dice"].mean(),
            "Precision":
                df["precision"].mean(),
            "Recall":
                df["recall"].mean()
        }
    )

# ------------------------------------------------------------
# Box + points
# ------------------------------------------------------------

for n in POINT_COUNTS:

    df = agriclip_box_point_dfs[n]

    comparison_rows.append(
        {
            "Method":
                f"AgriCLIP Box + {n} Points",
            "Prompt":
                f"K=32 box + top-{n} points",
            "IoU":
                df["iou"].mean(),
            "Dice":
                df["dice"].mean(),
            "Precision":
                df["precision"].mean(),
            "Recall":
                df["recall"].mean()
        }
    )

comparison_df = pd.DataFrame(
    comparison_rows
)

print("=" * 70)
print("PROMPT STRATEGY COMPARISON")
print("=" * 70)

display(
    comparison_df.style.format(
        {
            "IoU": "{:.4f}",
            "Dice": "{:.4f}",
            "Precision": "{:.4f}",
            "Recall": "{:.4f}"
        }
    )
)

PROMPT STRATEGY COMPARISON


,Method,Prompt,IoU,Dice,Precision,Recall
0,SAM2 + GT Box,Ground-truth box,0.8568,0.9175,0.9096,0.9330
1,SAM2 Automatic Prompting,Automatic grid points,0.8413,0.9035,0.8969,0.9205
2,AgriCLIP + SAM2,AgriCLIP K=32 box,0.8504,0.9117,0.9009,0.9341
3,AgriCLIP + SAM2 (1 points),Top-1 AgriCLIP points,0.8045,0.8782,0.9219,0.8571
4,AgriCLIP + SAM2 (3 points),Top-3 AgriCLIP points,0.7641,0.8484,0.9262,0.8101
5,AgriCLIP + SAM2 (5 points),Top-5 AgriCLIP points,0.7558,0.8414,0.9247,0.8017
6,AgriCLIP + SAM2 (8 points),Top-8 AgriCLIP points,0.7494,0.8378,0.9286,0.7911
7,AgriCLIP Box + 1 Points,K=32 box + top-1 points,0.8477,0.9105,0.9048,0.9259
8,AgriCLIP Box + 3 Points,K=32 box + top-3 points,0.8369,0.9030,0.9097,0.9086
9,AgriCLIP Box + 5 Points,K=32 box + top-5 points,0.8281,0.8964,0.9108,0.8974


In [82]:
# ============================================================
# CELL — EMPTY MASK IMPACT ON EXISTING AGRICLIP BOX RESULT
# ============================================================

all_results = []
nonempty_results = []
empty_results = []

K = 32

for idx in tqdm(
    range(len(val_hf)),
    desc="Checking empty-mask impact"
):

    image = get_image(
        idx
    )

    gt_mask = get_mask(
        idx
    )

    # --------------------------------------------------------
    # AgriCLIP box
    # --------------------------------------------------------

    loc = get_agriclip_localization(
        image,
        k=K
    )

    box = agriclip_patches_to_box(
        image,
        loc["top_indices"],
        loc["grid_h"],
        loc["grid_w"],
        margin=25
    )

    # --------------------------------------------------------
    # SAM2
    # --------------------------------------------------------

    output = sam2_box_prediction(
        image,
        box
    )

    metrics = compute_metrics_robust(
        output["prediction"],
        gt_mask
    )

    row = {
        "image_id": idx,
        "iou": metrics["iou"],
        "dice": metrics["dice"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "empty_gt": metrics["empty_gt"],
        "predicted_area": int(
            output["prediction"].sum()
        )
    }

    all_results.append(
        row
    )

    if metrics["empty_gt"]:
        empty_results.append(row)
    else:
        nonempty_results.append(row)


all_df = pd.DataFrame(
    all_results
)

nonempty_df = pd.DataFrame(
    nonempty_results
)

empty_df = pd.DataFrame(
    empty_results
)

print("\n" + "=" * 70)
print("EMPTY-MASK IMPACT")
print("=" * 70)

print(
    "All 755 images IoU:",
    f"{100 * all_df['iou'].mean():.2f}%"
)

print(
    "746 non-empty images IoU:",
    f"{100 * nonempty_df['iou'].mean():.2f}%"
)

print(
    "9 empty images IoU:",
    f"{100 * empty_df['iou'].mean():.2f}%"
)

print(
    "\nDifference:",
    f"{100 * (nonempty_df['iou'].mean() - all_df['iou'].mean()):+.2f} pp"
)

print("=" * 70)

Checking empty-mask impact:   0%|          | 0/755 [00:00<?, ?it/s]


EMPTY-MASK IMPACT
All 755 images IoU: 84.03%
746 non-empty images IoU: 85.04%
9 empty images IoU: 0.00%

Difference: +1.01 pp


In [91]:
# ============================================================
# CELL 36 — IMPROVEMENT RELATIVE TO AGRICLIP BOX BASELINE
# ============================================================

baseline_iou = (
    agriclip_box_df["iou"].mean()
)

comparison_df[
    "IoU_gain_pp"
] = (
    comparison_df["IoU"]
    - baseline_iou
) * 100

print("=" * 70)
print("IMPROVEMENT OVER AGRICLIP BOX BASELINE")
print("=" * 70)

display(
    comparison_df[
        [
            "Method",
            "IoU",
            "IoU_gain_pp",
            "Dice"
        ]
    ]
    .sort_values(
        "IoU",
        ascending=False
    )
    .style.format(
        {
            "IoU": "{:.4f}",
            "IoU_gain_pp": "{:+.2f}",
            "Dice": "{:.4f}"
        }
    )
)

IMPROVEMENT OVER AGRICLIP BOX BASELINE


,Method,IoU,IoU_gain_pp,Dice
0,SAM2 + GT Box,0.8568,+0.64,0.9175
2,AgriCLIP + SAM2,0.8504,+0.00,0.9117
7,AgriCLIP Box + 1 Points,0.8477,-0.27,0.9105
1,SAM2 Automatic Prompting,0.8413,-0.91,0.9035
8,AgriCLIP Box + 3 Points,0.8369,-1.35,0.9030
9,AgriCLIP Box + 5 Points,0.8281,-2.23,0.8964
10,AgriCLIP Box + 8 Points,0.8201,-3.03,0.8911
3,AgriCLIP + SAM2 (1 points),0.8045,-4.60,0.8782
4,AgriCLIP + SAM2 (3 points),0.7641,-8.63,0.8484
5,AgriCLIP + SAM2 (5 points),0.7558,-9.46,0.8414


In [92]:
# ============================================================
# CELL 37 — SAVE PROMPT EXPERIMENT RESULTS
# ============================================================

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "prompt_strategy_experiments"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

gt_df.to_csv(
    RESULTS_DIR /
    "sam2_gt_box.csv",
    index=False
)

automatic_df.to_csv(
    RESULTS_DIR /
    "sam2_automatic_prompting.csv",
    index=False
)

agriclip_box_df.to_csv(
    RESULTS_DIR /
    "agriclip_box_sam2_original_finetuned.csv",
    index=False
)

for n, df in agriclip_point_dfs.items():

    df.to_csv(
        RESULTS_DIR /
        f"agriclip_points_{n}.csv",
        index=False
    )

for n, df in agriclip_box_point_dfs.items():

    df.to_csv(
        RESULTS_DIR /
        f"agriclip_box_points_{n}.csv",
        index=False
    )

comparison_df.to_csv(
    RESULTS_DIR /
    "prompt_strategy_comparison.csv",
    index=False
)

print("=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(
    RESULTS_DIR
)

print("=" * 70)

RESULTS SAVED
/home/zahranadeem/thesis/results/prompt_strategy_experiments
